# Seminar 6

In [ ]:
book = open('crime_and_punishment.txt').read()
book = '***'.join(book.split('***')[2:-2])
book[:1000]

'\n\n\n\n\nCRIME AND PUNISHMENT\n\nBy Fyodor Dostoevsky\n\n\n\nTranslated By Constance Garnett\n\n\n\n\nTRANSLATOR’S PREFACE\n\nA few words about Dostoevsky himself may help the English reader to\nunderstand his work.\n\nDostoevsky was the son of a doctor. His parents were very hard-working\nand deeply religious people, but so poor that they lived with their five\nchildren in only two rooms. The father and mother spent their evenings\nin reading aloud to their children, generally from books of a serious\ncharacter.\n\nThough always sickly and delicate Dostoevsky came out third in the\nfinal examination of the Petersburg school of Engineering. There he had\nalready begun his first work, “Poor Folk.”\n\nThis story was published by the poet Nekrassov in his review and\nwas received with acclamations. The shy, unknown youth found himself\ninstantly something of a celebrity. A brilliant and successful career\nseemed to open before him, but those hopes were soon dashed. In 1849 he\nwas arres

In [ ]:
import re
text = re.findall('[A-Za-z]+', book)

In [ ]:
text[:10]

['CRIME',
 'AND',
 'PUNISHMENT',
 'By',
 'Fyodor',
 'Dostoevsky',
 'Translated',
 'By',
 'Constance',
 'Garnett']

## n-gram model

Let's start with unigram model:

$P(w_1, w_2, ..., w_m) = \prod_{i=1}^m P(w_i)$



In [ ]:
import numpy as np

dictionary, counts = np.unique(text, return_counts = True)
proba = counts / counts.sum()

In [ ]:
len(dictionary)

10125

In [ ]:
proba.sum()

1.0

In [ ]:
dictionary[2223], proba[2223], counts[2223]

('calmer', 4.784208285291908e-06, 1)

In [ ]:
dictionary[0], proba[0], counts[0]

('A', 0.0009233521990613384, 193)

Let's try to generate 30 words:

In [ ]:
np.random.seed(0)
np.random.choice(dictionary, size=30, p=proba)

array(['man', 'sent', 'observed', 'mad', 'her', 'out', 'him', 'trifling',
       'with', 'had', 'that', 'light', 'more', 'was', 'Petrovitch', 'So',
       'German', 'them', 'tears', 'to', 'yet', 'the', 'ideas', 'terror',
       'a', 'or', 'against', 'whatever', 'later', 'he'], dtype='<U18')

Problems:


*   A lot of stop-words
*   Words do not agree ('of she', 'sent observed')
*   Do not make sense


(If there were punctuation - a lot of it, if words from other languages - their mix)

Now, let's check bigger n,  n > 1



Let's use `nltk`

In [ ]:
text[:5]

['CRIME', 'AND', 'PUNISHMENT', 'By', 'Fyodor']

In [ ]:
from nltk.util import ngrams

list(ngrams(text[:5], 2))

[('CRIME', 'AND'),
 ('AND', 'PUNISHMENT'),
 ('PUNISHMENT', 'By'),
 ('By', 'Fyodor')]

In [ ]:
ngrams(text[:5], 2)

<generator object ngrams at 0x7de9b6f09360>

In [ ]:
list(ngrams(text[:5], 3))

[('CRIME', 'AND', 'PUNISHMENT'),
 ('AND', 'PUNISHMENT', 'By'),
 ('PUNISHMENT', 'By', 'Fyodor')]

In [ ]:
list(ngrams(text[:5], 4))

[('CRIME', 'AND', 'PUNISHMENT', 'By'), ('AND', 'PUNISHMENT', 'By', 'Fyodor')]

In [ ]:
from nltk.util import everygrams
list(everygrams(text[:5], max_len=3))

[('CRIME',),
 ('CRIME', 'AND'),
 ('CRIME', 'AND', 'PUNISHMENT'),
 ('AND',),
 ('AND', 'PUNISHMENT'),
 ('AND', 'PUNISHMENT', 'By'),
 ('PUNISHMENT',),
 ('PUNISHMENT', 'By'),
 ('PUNISHMENT', 'By', 'Fyodor'),
 ('By',),
 ('By', 'Fyodor'),
 ('Fyodor',)]

In [ ]:
len(text)

209021

In [ ]:
n = 3
list(everygrams(text, max_len=n))[:10]

[('CRIME',),
 ('CRIME', 'AND'),
 ('CRIME', 'AND', 'PUNISHMENT'),
 ('AND',),
 ('AND', 'PUNISHMENT'),
 ('AND', 'PUNISHMENT', 'By'),
 ('PUNISHMENT',),
 ('PUNISHMENT', 'By'),
 ('PUNISHMENT', 'By', 'Fyodor'),
 ('By',)]

Model - Maximum Likelihood Estimator (MLE)

In [ ]:
from nltk.lm import MLE
model = MLE(n)

In [ ]:
len(model.vocab)

0

Training:  n-grams (train_data)  and vocabulary (text)


what is  padded? Answer later

In [ ]:
from nltk.lm.preprocessing import padded_everygram_pipeline
train_data, padded_sents = padded_everygram_pipeline(n, [text])


model.fit(train_data, text)

In [ ]:
len(model.vocab)

10126

In [ ]:
list(model.vocab)[:10]

['CRIME',
 'AND',
 'PUNISHMENT',
 'By',
 'Fyodor',
 'Dostoevsky',
 'Translated',
 'Constance',
 'Garnett',
 'TRANSLATOR']

In [ ]:
model.counts['calmer']

1

In [ ]:
model.counts['A']

193

In [ ]:
model.counts['student']

44

Which words go after the particular one:

In [ ]:
model.counts[['Rodion']]

FreqDist({'Romanovitch': 86, 'Pray': 1, 'nothing': 1, 'in': 1, 'for': 1, 'and': 1, 'He': 1, 'but': 1, 'You': 1, 'There': 1, ...})

Count n-gram:

`model.counts[['Rodion']]['Romanovitch']` $=\#(Rodion,Romanovitch)$

In [ ]:
model.counts[['Rodion']]['Romanovitch'], model.counts[['Romanovitch']]['Rodion']

(86, 1)

In [ ]:
model.counts[['Rodion', 'Romanovitch']]

FreqDist({'I': 12, 'that': 7, 'Raskolnikov': 6, 'my': 5, 'you': 4, 'You': 3, 'And': 3, 'if': 2, 'Porfiry': 2, 'Yes': 2, ...})

In [ ]:
model.counts[['Rodion', 'Romanovitch']]['Raskolnikov']

6

Probability:

In [ ]:
model.score('Rodion')

0.0004640593230474824

In [ ]:
model.score('calmer')

4.784116732448272e-06

In [ ]:
model.score('A')

0.0009233345293625164


`model.score('Romanovitch', ['Rodion'])`  $=P(Romanovitch|Rodion)$

`model.score('Raskolnikov', ['Rodion', 'Romanovitch'])`  $=P(Raskolnikov|Rodion, Romanovitch)$



In [ ]:
model.score('Romanovitch', ['Rodion']), model.score('Raskolnikov', ['Rodion'])

(0.8865979381443299, 0.010309278350515464)

In [ ]:
model.score('Raskolnikov', ['Rodion', 'Romanovitch'])

0.06976744186046512

In [ ]:
model.score('I', ['Rodion', 'Romanovitch'])

0.13953488372093023

In [ ]:
model.score('Petrovitch', ['Rodion'])

0.0

In [ ]:
model.score('esrkl;gjshuierhgshirueg', ['Rodion'])

0.0

$P(w_1|w_2) \neq P(w_2|w_1)$

In [ ]:
model.score('Romanovitch', ['Rodion']) , model.score('Rodion', ['Romanovitch'])

(0.8865979381443299, 0.011627906976744186)

Generate text:

In [ ]:
#10 tokens
print(model.generate(10, random_seed=7))

['entered', 'Porfiry', 'Petrovitch', 'Yes', 'I', 'hear', 'and', 'have', 'called', 'on']


In [ ]:
#100 tokens
print(model.generate(100, random_seed=7))

['entered', 'Porfiry', 'Petrovitch', 'Yes', 'I', 'hear', 'and', 'have', 'called', 'on', 'Raskolnikov', 'again', 'He', 'was', 'afraid', 'of', 'me', 'that', 's', 'it', 'that', 'all', 'those', 'dark', 'mysterious', 'rumours', 'that', 'were', 'current', 'about', 'me', 'So', 'they', 'are', 'You', 'are', 'not', 'everything', 'at', 'that', 'moment', 'but', 'it', 'was', 'answered', 'by', 'a', 'sudden', 'she', 'is', 'your', 'decision', 'Rodya', 'asked', 'Avdotya', 'Romanovna', 'Confound', 'it', 'he', 'remembered', 'the', 'dreams', 'he', 'had', 'never', 'met', 'him', 'were', 'loathsome', 'to', 'her', 'long', 'black', 'eyelashes', 'were', 'quivering', 'as', 'though', 'I', 'hadn', 't', 'What', 'do', 'you', 'feel', 'about', 'female', 'faces', 'but', 'to', 'keep', 'up', 'with', 'rags', 'of', 'all', 'kinds', 'were', 'crowding', 'round']


In [ ]:
#10 000 tokens
print(model.generate(10_000, random_seed=7))

['entered', 'Porfiry', 'Petrovitch', 'Yes', 'I', 'hear', 'and', 'have', 'called', 'on', 'Raskolnikov', 'again', 'He', 'was', 'afraid', 'of', 'me', 'that', 's', 'it', 'that', 'all', 'those', 'dark', 'mysterious', 'rumours', 'that', 'were', 'current', 'about', 'me', 'So', 'they', 'are', 'You', 'are', 'not', 'everything', 'at', 'that', 'moment', 'but', 'it', 'was', 'answered', 'by', 'a', 'sudden', 'she', 'is', 'your', 'decision', 'Rodya', 'asked', 'Avdotya', 'Romanovna', 'Confound', 'it', 'he', 'remembered', 'the', 'dreams', 'he', 'had', 'never', 'met', 'him', 'were', 'loathsome', 'to', 'her', 'long', 'black', 'eyelashes', 'were', 'quivering', 'as', 'though', 'I', 'hadn', 't', 'What', 'do', 'you', 'feel', 'about', 'female', 'faces', 'but', 'to', 'keep', 'up', 'with', 'rags', 'of', 'all', 'kinds', 'were', 'crowding', 'round', 'Luzhin', 'with', 'all', 'its', 'consequences', 'They', 'listened', 'eagerly', 'to', 'right', 'and', 'to', 'my', 'mother', 'or', 'Dounia', 'She', 'was', 'terribly', '

Problems:



*   When to stop?
*   Whith what word to start?
*   Unknown words?



Unknown words - token `<UNK>`:

In [ ]:
model.counts['NLP']

0

In [ ]:
model.score("NLP")

1.9136466929793087e-05

In [ ]:
model.vocab.lookup('Rodion NLP Romanovitch'.split())

('Rodion', '<UNK>', 'Romanovitch')

In [ ]:
model.score("<UNK>") == model.score("NLP")

True

In [ ]:
model.score("<UNK>")

1.9136466929793087e-05

Start/end of text - tokens `<s>`, `</s>`

In [ ]:
padded_everygrams, padded_chain = padded_everygram_pipeline(3, [['the', 'cat', 'sat', 'on'],
                                                                ['One', 'two', 'three']])

for ngramlize_sent in padded_everygrams:
    print(list(ngramlize_sent))

#list(padded_chain)

[('<s>',), ('<s>', '<s>'), ('<s>', '<s>', 'the'), ('<s>',), ('<s>', 'the'), ('<s>', 'the', 'cat'), ('the',), ('the', 'cat'), ('the', 'cat', 'sat'), ('cat',), ('cat', 'sat'), ('cat', 'sat', 'on'), ('sat',), ('sat', 'on'), ('sat', 'on', '</s>'), ('on',), ('on', '</s>'), ('on', '</s>', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]
[('<s>',), ('<s>', '<s>'), ('<s>', '<s>', 'One'), ('<s>',), ('<s>', 'One'), ('<s>', 'One', 'two'), ('One',), ('One', 'two'), ('One', 'two', 'three'), ('two',), ('two', 'three'), ('two', 'three', '</s>'), ('three',), ('three', '</s>'), ('three', '</s>', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]


In [ ]:
model = MLE(3)
padded_everygrams, padded_chain = padded_everygram_pipeline(3, [['the', 'cat', 'sat', 'on'],
                                                                ['One', 'two', 'three']])
model.fit(padded_everygrams, padded_chain)

In [ ]:
model.counts[['<s>']]

FreqDist({'<s>': 2, 'the': 1, 'One': 1})

Example:

In [ ]:
np.random.seed(0)

synt_data = []
possible_start_tokens = ['foo', 'bar']
middle_token = '!'

for i in range(100):
  t = (list(np.random.choice(possible_start_tokens, size=1)) + #1 out of 2 tokens
       [middle_token] * np.random.randint(low=1, high=10)) #some middle tokens

  synt_data.append(t)

In [ ]:
synt_data[-10:]

[['foo', '!', '!', '!', '!', '!', '!', '!'],
 ['bar', '!', '!', '!', '!'],
 ['bar', '!', '!', '!', '!', '!', '!', '!', '!', '!'],
 ['foo', '!', '!', '!', '!', '!'],
 ['bar', '!', '!', '!', '!', '!', '!', '!'],
 ['bar', '!', '!', '!', '!', '!', '!', '!', '!'],
 ['foo', '!', '!', '!', '!', '!', '!', '!', '!', '!'],
 ['bar', '!', '!', '!'],
 ['foo', '!', '!', '!', '!', '!', '!', '!'],
 ['bar', '!', '!', '!', '!', '!', '!', '!']]

In [ ]:
model = MLE(3)
train_data, padded_sents = padded_everygram_pipeline(3, synt_data)

model.fit(train_data, padded_sents)

In [ ]:
model.counts[['</s>']]

FreqDist({'</s>': 100})

In [ ]:
print(model.generate(6, random_seed=0))

['<s>', 'bar', '!', '!', '!', '!']


In [ ]:
print(model.generate(12, random_seed=0))

['<s>', 'bar', '!', '!', '!', '!', '</s>', '</s>', '</s>', '</s>', '</s>', '</s>']


Other possible models nltk:

Lidstone smoothing:

$P(w_n | w_{n-1}, \dots, w_1) = \frac{C(w_1, \dots, w_n) + \gamma}{C(w_1, \dots, w_{n-1}) + \gamma \cdot V}$

$0 < \gamma < 1$


Laplace smoothing:

$P(w_n | w_{n-1}, \dots, w_1) = \frac{C(w_1, \dots, w_n) + 1}{C(w_1, \dots, w_{n-1}) + V}$


$V$ - size of the vocab

In [ ]:
from nltk.lm import Lidstone, Laplace

In [ ]:
lid = Lidstone(order=2, gamma=0.5)
lap = Laplace(2)

In [ ]:
train_data, padded_sents = padded_everygram_pipeline(n, [text])
lid.fit(train_data, text)
train_data, padded_sents = padded_everygram_pipeline(n, [text])
lap.fit(train_data, text)

 `score` for bigrams not in the text is not 0:

In [ ]:
lid.score('Petrovitch', ['Rodion']), lap.score('Petrovitch', ['Rodion'])

(9.689922480620155e-05, 9.781864423359092e-05)

Generate text:

In [ ]:
print(lid.generate(10, random_seed=7))

['end', 'Shall', 'I', 'am', 'mad', 'When', 'I', 'll', 'be', 'how']


In [ ]:
print(lap.generate(10, random_seed=7))

['else', 'It', 'was', 'a', 'millionth', 'fraction', 'of', 'lodgers', 'Amalia', 'Ivanovna']


# LSTM language model

char-based model

In [ ]:
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
TRAIN_TEXT_FILE_PATH = 'crime_and_punishment.txt'


with open(TRAIN_TEXT_FILE_PATH) as text_file:
    text_sample = text_file.readlines()
text_sample = ' '.join(text_sample)


def text_to_seq(text_sample):
    char_counts = Counter(text_sample)
    char_counts = sorted(char_counts.items(), key = lambda x: x[1], reverse=True)

    sorted_chars = [char for char, _ in char_counts]
    char_to_idx = {char: index for index, char in enumerate(sorted_chars)}
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    sequence = np.array([char_to_idx[char] for char in text_sample])

    return sequence, char_to_idx, idx_to_char

sequence, char_to_idx, idx_to_char = text_to_seq(text_sample)

In [ ]:
idx_to_char[5]

'n'

In [ ]:
char_to_idx['n']

5

In [ ]:
sequence

array([93, 35,  7, ..., 13,  0, 13])

In [ ]:
for idx in sequence[:20]:
  print(idx_to_char[idx])

﻿
T
h
e
 
P
r
o
j
e
c
t
 
G
u
t
e
n
b
e


In [ ]:
SEQ_LEN = 256
BATCH_SIZE = 16

def get_batch(sequence):
    trains = []
    targets = []
    for _ in range(BATCH_SIZE):
        batch_start = np.random.randint(0, len(sequence) - SEQ_LEN)
        chunk = sequence[batch_start: batch_start + SEQ_LEN]
        train = torch.LongTensor(chunk[:-1]).view(-1, 1)
        target = torch.LongTensor(chunk[1:]).view(-1, 1)
        trains.append(train)
        targets.append(target)
    return torch.stack(trains, dim=0), torch.stack(targets, dim=0)

In [ ]:
def evaluate(model, char_to_idx, idx_to_char, start_text=' ', prediction_len=200, temp=0.3):
    hidden = model.init_hidden()
    idx_input = [char_to_idx[char] for char in start_text]
    train = torch.LongTensor(idx_input).view(-1, 1, 1).to(device)
    predicted_text = start_text

    _, hidden = model(train, hidden)

    inp = train[-1].view(-1, 1, 1)

    for i in range(prediction_len):
        output, hidden = model(inp.to(device), hidden)
        output_logits = output.cpu().data.view(-1)
        p_next = F.softmax(output_logits / temp, dim=-1).detach().cpu().data.numpy()
        top_index = np.random.choice(len(char_to_idx), p=p_next)
        inp = torch.LongTensor([top_index]).view(-1, 1, 1).to(device)
        predicted_char = idx_to_char[top_index]
        predicted_text += predicted_char

    return predicted_text

In [ ]:
class TextRNN(nn.Module):

    def __init__(self, input_size, hidden_size, embedding_size, n_layers=1):
        super(TextRNN, self).__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size
        self.n_layers = n_layers

        self.encoder = nn.Embedding(self.input_size, self.embedding_size)
        self.lstm = nn.LSTM(self.embedding_size, self.hidden_size, self.n_layers)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.hidden_size, self.input_size)

    def forward(self, x, hidden):
        x = self.encoder(x).squeeze(2)
        out, (ht1, ct1) = self.lstm(x, hidden)
        out = self.dropout(out)
        x = self.fc(out)
        return x, (ht1, ct1)

    def init_hidden(self, batch_size=1):
        return (torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device),
               torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device))

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = TextRNN(input_size=len(idx_to_char), hidden_size=128, embedding_size=128, n_layers=2)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=5,
    verbose=True,
    factor=0.5
)

n_epochs = 3000
loss_avg = []

for epoch in range(n_epochs):
    print(epoch)
    model.train()
    train, target = get_batch(sequence)
    train = train.permute(1, 0, 2).to(device)
    target = target.permute(1, 0, 2).to(device)
    hidden = model.init_hidden(BATCH_SIZE)

    output, hidden = model(train, hidden)
    loss = criterion(output.permute(1, 2, 0), target.squeeze(-1).permute(1, 0))

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    loss_avg.append(loss.item())
    if len(loss_avg) >= 5:
        mean_loss = np.mean(loss_avg)
        print(f'Loss: {mean_loss}')
        scheduler.step(mean_loss)
        loss_avg = []
        model.eval()
        predicted_text = evaluate(model, char_to_idx, idx_to_char)
        print(predicted_text)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


0
1
2
3
4
Loss: 3.7262850761413575
  e  er    e   e e e o  o           e   e      ae e   o    t e    ie o ote e e    i     e    eoae       e e   e   e       e    e eee        e    o     e   o t  e e e e e   ea  e o eee      e o     t ae
5
6
7
8
9
Loss: 3.1725400924682616
 sa sm  t a a hs tta a a te a o hi t  aa aa ata e a a h a at e aa s ae s aa ta a a e ao  lo haiaio  ah  han a  o    hoa a fe ad ta a t a a s  ae ap o oho a   t aai  aaaah a att a e i a ad i t aa o a a 
10
11
12
13
14
Loss: 3.025563192367554
 e a has e he he e ie toee sa e et te ne no e e e e ehn e te te as ho ae e e oe  te he o a oe ti e te oe o te o ait te hie as ee he te te oe aa tae ie te to te te hie e het ie ta ae he o o he e o e o e
15
16
17
18
19
Loss: 2.90445556640625
 ee po an a hot ta to a har ae a ae a eit a he tat as ho ei ar an o ae por ht ol ta od a o hooe ae tae the au a tan an a ho ae tho a ta on to hod e he a to a a an ho on or ii on on e oae an aa an a en 
20
21
22
23
24
Loss: 2.787452745437622
 an an

KeyboardInterrupt: 

In [ ]:
model.eval()

print(evaluate(
    model,
    char_to_idx,
    idx_to_char,
    temp=0.3,
    prediction_len=80,
    start_text='Rodion'
    )
)

Rodioned and and and and the beon and the with a fear and sort and and the stree to th


# Transformers library

[Hugging Face](https://huggingface.co/) -  for transformers, datasets, etc.

In [ ]:
!pip install --quiet datasets evaluate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.6 MB/s eta 0:00:00


`pipeline` - easy way to use pretrained model


Documentation: https://huggingface.co/docs/transformers/v4.45.2/en/main_classes/pipelines#transformers.pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(["The film is so cool!",
            "I hate my scool."])

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998786449432373},
 {'label': 'NEGATIVE', 'score': 0.9993107318878174}]

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier(["The film is so cool!",
            "I hate my scool."])

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998786449432373},
 {'label': 'NEGATIVE', 'score': 0.9993107318878174}]

In [ ]:
classifier(["Cat"])

[{'label': 'POSITIVE', 'score': 0.9852107167243958}]

Other tasks:

In [ ]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about NLP",
    candidate_labels=["education", "media", "business"],
)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


{'sequence': 'This is a course about NLP',
 'labels': ['education', 'media', 'business'],
 'scores': [0.7910552620887756, 0.14676831662654877, 0.062176384031772614]}

In [ ]:
classifier(
    "This is a course about NLP",
    candidate_labels=["education", "video games", "TV show"],
)

{'sequence': 'This is a course about NLP',
 'labels': ['education', 'video games', 'TV show'],
 'scores': [0.9365969896316528, 0.03402788192033768, 0.02937513403594494]}

MLM task:

In [ ]:
unmasker = pipeline("fill-mask")
unmasker("I like <mask> and burgers.", top_k=2)

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


[{'score': 0.08730415254831314,
  'token': 22391,
  'token_str': ' fries',
  'sequence': 'I like fries and burgers.'},
 {'score': 0.08562362194061279,
  'token': 19464,
  'token_str': ' steak',
  'sequence': 'I like steak and burgers.'}]

In [ ]:
unmasker("This is a course about <mask> language processing", top_k=10)

[{'score': 0.4299525320529938,
  'token': 1632,
  'token_str': ' natural',
  'sequence': 'This is a course about natural language processing'},
 {'score': 0.04939333349466324,
  'token': 12628,
  'token_str': ' functional',
  'sequence': 'This is a course about functional language processing'},
 {'score': 0.039077069610357285,
  'token': 3563,
  'token_str': ' machine',
  'sequence': 'This is a course about machine language processing'},
 {'score': 0.03516726568341255,
  'token': 8326,
  'token_str': ' programming',
  'sequence': 'This is a course about programming language processing'},
 {'score': 0.026512475684285164,
  'token': 1050,
  'token_str': ' human',
  'sequence': 'This is a course about human language processing'},
 {'score': 0.024829326197504997,
  'token': 16868,
  'token_str': ' symbolic',
  'sequence': 'This is a course about symbolic language processing'},
 {'score': 0.024130333214998245,
  'token': 7350,
  'token_str': ' artificial',
  'sequence': 'This is a course ab

BIAS:

In [ ]:
#Internet texts...
unmasker = pipeline("fill-mask", model="bert-base-uncased")
result = unmasker("This man works as a [MASK].")
print([r["token_str"] for r in result])

result = unmasker("This woman works as a [MASK].")
print([r["token_str"] for r in result])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


['carpenter', 'lawyer', 'farmer', 'businessman', 'doctor']
['nurse', 'maid', 'teacher', 'waitress', 'prostitute']


NER:

In [ ]:
ner = pipeline("ner", grouped_entities=True)
ner("My name is Sasha, I work at HSE and I live in Moscow. I like pizza and burgers.")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Device set to use cpu
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/token_classification.py:170: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


[{'entity_group': 'PER',
  'score': 0.9978769,
  'word': 'Sasha',
  'start': 11,
  'end': 16},
 {'entity_group': 'ORG',
  'score': 0.99515545,
  'word': 'HSE',
  'start': 28,
  'end': 31},
 {'entity_group': 'LOC',
  'score': 0.9996026,
  'word': 'Moscow',
  'start': 46,
  'end': 52}]

Translation:

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")
#This is from Google:
translator("J'adore les pizzas et les hamburgers")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


[{'translation_text': 'I love pizza and burgers.'}]

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")

translator("Cat sat on the mat")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

Device set to use cpu


[{'translation_text': 'Chat assis sur le tapis'}]

Text generation:

In [ ]:
generator = pipeline("text-generation")
generator("I want to eat pizza and")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'I want to eat pizza and have a pizza. We were eating some pizza at the counter and he came running from the kitchen with pizza. I think I probably get $15 to eat at my first pizza or at someplace."\n\n"You'}]

In [ ]:
generator("I want to eat pizza and")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'I want to eat pizza and I\'ve done that already," Parecchi said. "I just wanted to be safe and not be hurt."\n\nThe first time Parecchi saw another pizza coming out was a week earlier this year'}]

In [ ]:
generator("I want to eat pizza and")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'I want to eat pizza and make it all go well, and that\'s because pizza was pretty wonderful in my time with it."\n\nThe couple and others who attended the event had nothing but praise for the pizza on Twitter. They added that the'}]

In [ ]:
generator("This man works as a")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'This man works as a trucker, and I think this is just the latest example of his skills."The video begins in the southern French city of Bordeaux and goes on to show a man trying to get some work done while trying to walk'}]

In [ ]:
generator("This woman works as a")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': "This woman works as a nurse at the Children's Health Service. She got the medical degree at the University of Washington. And she works in public health as a nurse. And her husband, David, who is a doctor, is with them as their"}]

In [ ]:
generator("This woman works as a")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': "This woman works as a delivery girl and her job is to provide social service to those in need. She is also a community worker and has a bachelor's degree. She has an undergraduate degree in education and has work experience in local social services. ["}]

Default model may be bad for our language/task

We may search for particular model : https://huggingface.co/models



In [ ]:
generator = pipeline("text-generation", model='lysandre/arxiv-nlp')
generator(
    "paper",
    max_length=30
)

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'paper, vol. 3, no. 2, pp. 535–543, 2005.\n\n[5] R. S. S'}]

In [ ]:
generator(
    "paper",
    max_length=30
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'paper, vol. 3, no. 2, pp. 535–543, 2005.\n\n[5] R. S. S'}]

In [ ]:
generator(
    "natural"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'natural, and the most important of all, the most important of all, the most important of all,'}]

In [ ]:
generator(
    "language"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'language, and the language of the target language.\n\nWe use the following language model to model the'}]

Add arguments:

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")
generator(
    "I want to eat pizza and",
    max_length=30,
    num_return_sequences=2 #Not for all models
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'I want to eat pizza and pizza," he said, referring to the recent controversy surrounding a pizza truck driver\u200fs allegedly breaking open windows in Houston'},
 {'generated_text': "I want to eat pizza and enjoy it, so feel free to ask us if that's the person you have chosen to meet, if you've got"}]

**Sources**
 https://huggingface.co/learn/nlp-course/chapter1/1

